## TASK 2 Data Analysis ##

In [1]:
import psycopg2

# Small helper class to centralize PostgreSQL connection handling.
# Why we do this:
# - Avoid repeating credentials/connection logic in every cell
# - Reuse a single connection (faster + fewer open connections)
# - Provide a single place to close the DB connection cleanly

class PostgresConnector:
    def __init__(
        self,
        dbname: str = "db_berlin",
        user: str = "dia_user",
        password: str = "dia",
        host: str = "localhost",
        port: int = 5434,
    ):
        # Store connection parameters (used later by psycopg2.connect)
        self.dbname = dbname
        self.user = user
        self.password = password
        self.host = host
        self.port = port
        self.connection: Optional[psycopg2.extensions.connection] = None

    def connect(self):
        if not self.connection:
            # Establish a TCP connection to PostgreSQL using the stored credentials.
            # psycopg2 returns a connection object that can create cursors and run queries.
            self.connection = psycopg2.connect(
                dbname=self.dbname,
                user=self.user,
                password=self.password,
                host=self.host,
                port=self.port
            )
        return self.connection

    def close(self):
        if self.connection:
            self.connection.close()
            self.connection = None

In [2]:
# Instantiate connector with default credentials/host/port
connector = PostgresConnector()

# Open (or reuse) a PostgreSQL connection
conn = connector.connect()

# Cursor is the object we use for executing SQL directly via psycopg2
# (Alternative approach used later: pandas.read_sql, which runs the query + returns DataFrame)
cur = conn.cursor()

In [3]:
import pandas as pd

# Simple validation query:
# If dim_stations has rows, then schema creation + station ingestion likely worked.
query = "SELECT count(*) FROM dim_stations"

# pandas.read_sql executes the SQL and returns the result as a DataFrame.
df = pd.read_sql(query, connector.connect())

print(df)

   count
0    133


/var/folders/j4/78mwfbr90r19ts01v8jq3hs40000gn/T/ipykernel_1229/2033433013.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, connector.connect())


### TASK 2.1 ### 

In [4]:
# TASK 2.1: Find specific station details
# Goal: Return metadata for a given station name ("Tiergarten")
# Tables used: dim_stations (dimension table containing station attributes)
query_2_1 = """
    SELECT 
        eva_number, 
        station_name, 
        latitude, 
        longitude
    FROM dim_stations
    WHERE station_name = 'Tiergarten';
"""
# Execute the SQL and store the result in a DataFrame for display
df_2_1 = pd.read_sql(query_2_1, connector.connect())
display(df_2_1)

/var/folders/j4/78mwfbr90r19ts01v8jq3hs40000gn/T/ipykernel_1229/3211450377.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_2_1 = pd.read_sql(query_2_1, connector.connect())


,eva_number,station_name,latitude,longitude
0,8089091,Tiergarten,52.514061,13.336392


### TASK 2.2 ### 

In [5]:
# TASK 2.2: Find closest station to coordinates
# Goal: Given a latitude/longitude, find the "closest" station.
# This uses a simple Euclidean-like distance in lat/long degrees.
# That is not true geodesic distance (Haversine would be more accurate), but it is acceptable for a rough comparison within a city.
query_2_2 = """
    SELECT 
        station_name,
        latitude,
        longitude,
        SQRT(POWER(latitude - 52.5200, 2) + POWER(longitude - 13.4050, 2)) as distance_metric
    FROM dim_stations
    ORDER BY distance_metric ASC
    LIMIT 1;
"""
df_2_2 = pd.read_sql(query_2_2, connector.connect())
display(df_2_2)

/var/folders/j4/78mwfbr90r19ts01v8jq3hs40000gn/T/ipykernel_1229/2118235975.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_2_2 = pd.read_sql(query_2_2, connector.connect())


,station_name,latitude,longitude,distance_metric
0,Hackescher Markt,52.522622,13.402364,0.003718


### TASK 2.3 ### 

In [6]:
# TASK 2.3: Total cancelled trains on a specific date/hour
# Goal: Count cancelled stop events for a given snapshot time:
# - date: 2025-09-02
# - hour: 11
#
# Why join dim_time?
# - fact_train_stops stores time_id (YYYYMMDDHH)
# - dim_time stores human-readable fields like full_date and hour
cur.execute("""
    SELECT 
        COUNT(*) as total_cancelled_trains
    FROM fact_train_stops f
    JOIN dim_time t ON f.time_id = t.time_id
    WHERE 
        t.full_date = '2025-09-02'
        AND t.hour = 11
        AND f.is_cancelled = TRUE;
""")

# fetchone() returns a tuple like (count_value,)
# so [0] extracts the integer count
cancelled_count = cur.fetchone()[0]
print(f"Total cancelled trains at 11:00 on 2025-09-02: {cancelled_count}")

Total cancelled trains at 11:00 on 2025-09-02: 1


### TASK 2.4 ### 

In [7]:
# TASK 2.4: Average delays for a specific station
# Goal: Compute average arrival and departure delay (in minutes)
# for a given station, excluding cancelled events.
#
# Why exclude cancellations?
# - Cancelled events may have missing/invalid actual times
# - Including them could distort delay averages
query_2_4 = """
    SELECT 
        s.station_name,
        ROUND(AVG(f.arrival_delay_min), 2) as avg_arrival_delay_minutes,
        ROUND(AVG(f.departure_delay_min), 2) as avg_departure_delay_minutes
    FROM fact_train_stops f
    JOIN dim_stations s ON f.station_eva_number = s.eva_number
    WHERE 
        s.station_name = 'Berlin Ostbahnhof'
        AND f.is_cancelled = FALSE
    GROUP BY s.station_name;
"""
df_2_4 = pd.read_sql(query_2_4, connector.connect())
display(df_2_4)

/var/folders/j4/78mwfbr90r19ts01v8jq3hs40000gn/T/ipykernel_1229/560239795.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_2_4 = pd.read_sql(query_2_4, connector.connect())


,station_name,avg_arrival_delay_minutes,avg_departure_delay_minutes
0,Berlin Ostbahnhof,4.14,2.02
